# Kokoro TextToSpeech (TTS) Generator and TXT cleaner.

This delightful little tool leverages  [Kokoro TTS](https://huggingface.co/onnx-community/Kokoro-82M-v1.0-ONNX) and the [QWEN Instruct](https://huggingface.co/Qwen/Qwen2.5-3B-Instruct) models to process .txt files within Google Colab, requiring absolutely no technical wizardry on your part.

Designed for exceptional simplicity, it transforms your reading material with just a few easy steps:

1. **Copy and Paste**: Gather your source text and paste it directly into Notepad, TextEdit, or your favorite text editor.

2. **Save**: Save the document as a standard .txt file. There is no need to fuss over formatting or tidy up the layout!

3. **Listen**: Run the file through this Jupyter Notebook, sit back, and let it spin your plain text into a beautiful, audiobook style masterpiece.

Consider it your personal, tireless narrator, ready to bring your documents to life at a moment's notice!

In [ ]:
# @title 🛠️ AI Text Cleaner & 🎙️ TTS Generator 🛠️
# @markdown ### Select your Task and Input Source below:
Task = "Both: Clean Text then Generate TTS" # @param ["Text Cleaner Only", "TTS Generator Only", "Both: Clean Text then Generate TTS"]
input_source = "Upload .txt File" # @param ["Text Box", "Upload .txt File"]
# @markdown <hr />

# @markdown ### ⚙️ General Settings:
# @markdown **save_to_google_drive:** Automatically save outputs to Google Drive? (Requires Login)
save_to_google_drive = False # @param {type:"boolean"}
# @markdown **text:** If inputting via 'Text Box', paste that here. Ignored for files.
text = "Put your text here, This box is ignored if you are using a .txt file." # @param {type:"string"}
# @markdown **output_filename:** Ignored for file uploads (uses input filename instead).
output_filename = "my_output" # @param {type:"string"}
# @markdown <hr />

# @markdown ### 🎙️ TTS Settings (Only used if generating audio):
# @markdown **voice:** Select the voice you wish to use. All valid entities and samples can be found [HERE](https://huggingface.co/onnx-community/Kokoro-82M-v1.0-ONNX#voicessamples) in the `Voices/Samples` section.<br />
# @markdown *Note:* Voice format is  a/b (American/British) f/m (Feminine/Masculine) _name, E.G. af_nova = An American, Feminine voice.
voice = "af_nova" # @param ["af_nova", "af_heart", "bf_emma", "am_fenrir", "bm_daniel"] {allow-input: true}
# @markdown <hr />


import os
import sys
import subprocess
import shutil
import re
from IPython.display import Audio, display, clear_output
from google.colab import files

# --- BYPASS HUGGING FACE TOKEN POPUP ---
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["HF_HUB_DISABLE_TOKEN_WARNING"] = "1"
# ---------------------------------------

# --- 1. GET THE INPUT TEXT ---
raw_input = ""
base_name = output_filename

if input_source == "Upload .txt File":
    print("📂 Awaiting file upload... Please select your .txt file below.")
    uploaded = files.upload()
    if not uploaded:
        print("❌ No file uploaded. Execution stopped.")
        sys.exit()

    original_filename = list(uploaded.keys())[0]
    raw_input = uploaded[original_filename].decode('utf-8')
    print(f"✅ Loaded '{original_filename}' successfully.")

    # Dynamically set base filename
    base_name = os.path.splitext(original_filename)[0]
else:
    raw_input = text

if not raw_input.strip():
    print("⚠️ No text detected. Please paste text in the box or upload a file.")
    sys.exit()

# This variable will hold our text as it moves through the pipeline
current_text = raw_input


# --- 2. TEXT CLEANER LOGIC ---
if "Text Cleaner" in Task or "Both" in Task:
    print("\n" + "="*50)
    print("🖨️ STARTING AI TEXT CLEANER...")
    print("="*50)

    # Smart Initialization (Only runs on first click)
    try:
        import transformers
    except ImportError:
        print("📦 Installing required libraries... (First run only)")
        subprocess.run(["pip", "install", "-q", "-U", "transformers", "accelerate", "torch"], check=True)

    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    if 'model' not in globals() or 'tokenizer' not in globals():
        print("⏳ Loading Qwen2.5-3B-Instruct into GPU... (Takes 1-2 minutes)")
        model_name = "Qwen/Qwen2.5-3B-Instruct"
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16,
            device_map="auto"
        )
        print("✅ Model loaded successfully!")
    else:
        print("⚡ Model already in memory. Skipping setup.")

    def process_text_with_ai(raw_text):
        system_prompt = (
            "You are an expert text editor. Your task is to clean up badly formatted text "
            "Fix random line breaks, broken hyphenations, weird spacing, and remove inline "
            "headers, footers, or page numbers, including lines that just contain a single number. "
            "Preserve the original meaning and structure. "
            "Output ONLY the cleaned text."
        )
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Please clean the following text:\n\n{raw_text}"}
        ]
        formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        model_inputs = tokenizer([formatted_prompt], return_tensors="pt").to(model.device)
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=2000,
            temperature=0.1,
            do_sample=True,
        )
        generated_ids = [
            output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        return tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

    final_txt_filename = f"{base_name}_cleaned.txt"
    print(f"✨ Starting AI cleaning process... Output will be saved to: {final_txt_filename}")

    # Process large documents in chunks
    chunk_size = 2500
    words = current_text.replace('\n', ' \n ').split(' ')
    chunks = []
    current_chunk = ""

    for word in words:
        if len(current_chunk) + len(word) + 1 < chunk_size:
            current_chunk += word + " "
        else:
            if current_chunk.strip():
                chunks.append(current_chunk.strip())
            current_chunk = word + " "
    if current_chunk.strip():
        chunks.append(current_chunk.strip())

    print(f"🧩 Document safely split into {len(chunks)} manageable chunks.")

    with open(final_txt_filename, "w", encoding="utf-8") as f:
        f.write("")

    for i, chunk in enumerate(chunks):
        print(f"⏳ Cleaning section {i+1} of {len(chunks)}...")
        cleaned_chunk = process_text_with_ai(chunk)

        with open(final_txt_filename, "a", encoding="utf-8") as f:
            f.write(cleaned_chunk + "\n\n")
            f.flush()
            os.fsync(f.fileno())

    print(f"🎉 Done! Cleaned text completely saved to: {final_txt_filename}")

    # Read the completely cleaned file back into memory for the TTS step
    with open(final_txt_filename, "r", encoding="utf-8") as f:
        current_text = f.read()

    if save_to_google_drive:
        from google.colab import drive
        if not os.path.exists('/content/drive/MyDrive'):
            print("Mounting Google Drive...")
            drive.mount('/content/drive')
        drive_path = f"/content/drive/MyDrive/{final_txt_filename}"
        shutil.copy(final_txt_filename, drive_path)
        print(f"💾 Successfully saved text to Google Drive at: {drive_path}")

    try:
        if "Both" in Task:
            print(f"⚠️ Text cleanup finished. Colab will queue the text download to process at the very end of the cell.")
        files.download(final_txt_filename)
    except Exception as e:
        print(f"⚠️ Could not trigger automatic download. Find '{final_txt_filename}' on the left menu.")


# --- 3. TTS GENERATOR LOGIC ---
if "TTS Generator" in Task or "Both" in Task:
    print("\n" + "="*50)
    print("🎙️ STARTING KOKORO TTS GENERATOR...")
    print("="*50)

    try:
        import kokoro
        import soundfile as sf
    except ImportError:
        print("📦 Installing PyTorch Kokoro and dependencies (takes a minute on first run)...")
        subprocess.run(["pip", "install", "-q", "kokoro", "soundfile"], check=True)
        clear_output()
        import soundfile as sf

    import torch
    import numpy as np
    from kokoro import KPipeline

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"🚀 Computing Device: {device.upper()}")
    if device == 'cpu':
        print("⚠️ WARNING: GPU not detected. Generation will be slow. Go to Runtime > Change runtime type > select T4 GPU.")

    # Format output wav filename based on whether we cleaned it or not
    final_wav_filename = f"{base_name}_cleaned.wav" if "Both" in Task else f"{base_name}.wav"

    lang_code = voice[0]
    print(f"\nLoading Kokoro-82M model onto {device.upper()}...")
    pipeline = KPipeline(lang_code=lang_code, device=device)

    print(f"Generating speech for voice: {voice}...")

    # Split into chunks for processing
    text_chunks = [chunk.strip() for chunk in re.split(r'(?<=[.!?\n])\s+', current_text) if chunk.strip()]
    total_chunks = len(text_chunks)
    audio_chunks = []

    for i, chunk_text in enumerate(text_chunks):
        generator = pipeline(chunk_text, voice=voice, speed=1.0)
        for graphemes, phonemes, audio in generator:
            audio_chunks.append(audio)
        print(f"  -> Processed chunk {i+1} out of {total_chunks}...")

    if audio_chunks:
        print("Merging chunks and saving file...")
        final_audio = np.concatenate(audio_chunks)

        sf.write(final_wav_filename, final_audio, 24000)
        print(f"✅ Successfully created: {final_wav_filename}")

        if save_to_google_drive:
            from google.colab import drive
            if not os.path.exists('/content/drive/MyDrive'):
                print("Mounting Google Drive...")
                drive.mount('/content/drive')
            drive_path = f"/content/drive/MyDrive/{final_wav_filename}"
            shutil.copy(final_wav_filename, drive_path)
            print(f"💾 Successfully saved audio to Google Drive at: {drive_path}")

        display(Audio(final_wav_filename, autoplay=True))

        # --- MISSING CODE ADDED BELOW ---
        print("\n📥 Triggering Audio Download...")
        if "Both" in Task:
            print("⚠️ NOTE: Your browser may ask for permission to download multiple files at once. Please click 'Allow' in your URL bar if prompted.")

        try:
            files.download(final_wav_filename)
        except Exception as e:
            print(f"⚠️ Could not trigger automatic download. Find '{final_wav_filename}' on the left menu.")

    else:
        print("❌ Error: Audio generation failed.")